In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#removed fine tune
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.applications import ConvNeXtBase
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping , ReduceLROnPlateau
from tensorflow.keras.applications.convnext import preprocess_input
from PIL import ImageFile
from sklearn.metrics import classification_report
import numpy as np

# Enable loading of truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Load and prepare the CSV data
df = pd.read_csv('/kaggle/input/indicleands/cleaned_indicleaf (1).csv')
df['filepath'] = df['filepath'].astype(str)
df['class_label'] = df['class_label'].astype(str)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split data into train and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class_label'])

# Create data generators with ConvNeXt preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# Generators
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='class_label',
    target_size=(224, 224),
    color_mode='rgb',
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='class_label',
    target_size=(224, 224),
    color_mode='rgb',
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Get number of classes
num_classes = len(train_generator.class_indices)

# Build ConvNeXtBase model
base_model = ConvNeXtBase(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the base model
base_model.trainable = False

# Add custom head
inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)

# Compile with higher learning rate
model.compile(
    optimizer=Adam(learning_rate=0.0001),  
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stop]
)

# Fine-tuning phase: unfreeze top layers
base_model.trainable = True
for layer in base_model.layers[:-80]:  # Unfreeze last 80 layers
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tuning callbacks
fine_tune_callbacks = [
    ModelCheckpoint(
        'fine_tuned_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=1e-7
    )
]

# Fine-tuning training
history_fine = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=fine_tune_callbacks
)

# Load best weights and save final model
model.load_weights('best_model.keras')
model.save('plant_classification_indic_convnext.keras')

# Evaluation
results = model.evaluate(val_generator)
print(f'Validation Loss: {results[0]}, Validation Accuracy: {results[1]}')

val_preds = model.predict(val_generator, verbose=1)
val_preds_class = np.argmax(val_preds, axis=1)

true_labels = val_generator.classes
label_map = {v: k for k, v in train_generator.class_indices.items()}
target_names = [label_map[i] for i in range(num_classes)]

report = classification_report(true_labels, val_preds_class, target_names=target_names)
print(report)

In [ ]:

from sklearn.preprocessing import LabelEncoder
import joblib

# Assume your CSV has a column 'class_label'
df = pd.read_csv('/kaggle/input/indicleands/cleaned_indicleaf (1).csv')

# Initialize and fit the encoder on your class labels
encoder = LabelEncoder()
df['encoded_label'] = encoder.fit_transform(df['class_label'])

# Optionally, you can save the encoder for later use in prediction
joblib.dump(encoder, 'label_encoder.pkl')


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import joblib

def predict_image(image_path, model_path, encoder_path=None):
    # Load the trained model (.keras format)
    model = load_model(model_path)
    
    # Load and preprocess the image
    img = image.load_img(image_path, target_size=(224, 224), color_mode='rgb')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict the class probabilities
    predictions = model.predict(img_array)
    
    # Get the index of the most dominant class
    dominant_index = np.argmax(predictions, axis=1)[0]
    
    # If an encoder path is provided, load the encoder and convert the index to the actual label
    if encoder_path is not None:
        encoder = joblib.load(encoder_path)
        dominant_label = encoder.inverse_transform([dominant_index])[0]
    else:
        dominant_label = dominant_index  # Fallback to numeric label if encoder is not provided
    
    return dominant_label, predictions

# Example usage
if __name__ == '__main__':
    image_path = '/kaggle/input/img3333/6b7284dfeab7222691aa7e7472d8d4bc.jpg'
    model_path = '/kaggle/input/plant-classification/keras/default/1/fine_tuned_model (1).keras'
    encoder_path = '/kaggle/working/label_encoder.pkl'  # Path to your saved encoder
    
    dominant_label, preds = predict_image(image_path, model_path, encoder_path)
    print("Predicted Class:", dominant_label)
